# Proje 3 — Training ve Metrik Karşılaştırma (Aşama 4)
Dua Lipa sesiyle iki RVC v2 modeli: **A** (sadece gerçek, 15.1 dk) ve **B** (gerçek + ayıklanmamış Seed-VC sentetiği, 15.1 + 8.9 dk).
Eğitim GPU'da yapıldı; bu notebook sunucunun CPU'sunda çeviri, Ses benzerliği ve oktav kontrolünü yapar. Hücreleri sırayla çalıştır.

**Ayarlar.** Klasörleri ve hedef şarkıyı tanımlar, yardımcıları (`vc_araclar.py`) yükler.
Çıktıda üç modelin klasörünün bulunduğunu (`True`) kontrol et. Final B, ilk çalıştırmada **ayıklanmamış** sentetikle (`synth/`, 8.9 dk) eğitildi; ayıklanmış B'nin eğitimi öğretmen tarafından iptal edildi.

In [ ]:
import glob, json
from pathlib import Path
import pandas as pd
from IPython.display import Audio, HTML, Image, display
import vc_araclar as v

HEDEF = "cover_simge_uzulmedin_mi"           # cover'ı yapılacak şarkı
HEDEF_VOKAL = f"stems/{HEDEF}_lead.wav"        # kuru ana vokal
KITA = (22.0, 10.0)                            # checkpoint dinleme: ilk kıtadan 10 sn (nakarat değil)
KAYDIRMA = 0                                   # Aşama 2'deki ses aralığı kararı
MODELLER = {"A": "models/A", "B": "models/B_ayiklanmamis"}  # B: öğretmen 2. eğitimi iptal etti, final B 1. çalıştırmadan
OUT = Path("outputs/asama4"); OUT.mkdir(parents=True, exist_ok=True)
for m, d in MODELLER.items():
    print(m, d, Path(d).is_dir(), len(v.checkpointler(d)), "checkpoint")

**Üç örnek.** Bir gerçek Dua Lipa train kaydı, bir tutulan sentetik parça (Seed-VC) ve hedef şarkının kuru vokali (Simge).
Model melodiyi ve kelimeleri üçüncüden, ses rengini birinciden almalı.

In [ ]:
print("Gerçek Dua Lipa (train)"); display(Audio(sorted(glob.glob("dataset_temiz/train/dua_love_again_*.wav"))[5]))
print("Sentetik parça (Seed-VC, tutuldu)"); display(Audio(sorted(glob.glob("synth_keep/*.wav"))[0]))
print("Hedef: Simge – Üzülmedin mi, kuru vokal (ilk kıta)"); display(Audio(v.kes(HEDEF_VOKAL, OUT / "hedef_kita.wav", *KITA)))

**Ses aralığı (Aşama 2).** Dua Lipa train ve hedef şarkıların nota histogramları. Ortancalar 2 yarım ton içinde, bu yüzden kaydırma **0**.

In [ ]:
display(Image("outputs/ses_araligi.png"))
print("seçilen kaydırma:", KAYDIRMA)

**Altın kural: assert.** Eğitim listelerinde (gerçek: `dataset_temiz/train/`, B'nin sentetiği: `synth/`) test şarkısı (Pretty Please) ve cover şarkıları (Simge) olmamalı.
Hücre hata vermeden `assert geçti` yazmalı. Ayrıca GPU'daki dilim sayıları (`timings`) yazdırılır: A ile B'nin adım sayısı aynı olmalı.

In [ ]:
real = sorted(Path(p).name for p in glob.glob("dataset_temiz/train/*.wav"))
synth = sorted(Path(p).name for p in glob.glob("synth/*.wav"))
test = {Path(p).name for p in glob.glob("dataset_temiz/test/*.wav")}
egitim = real + synth
assert real and synth, "eğitim listeleri boş"
assert not test & set(egitim), "test dosyası eğitimde!"
assert not any("pretty_please" in f for f in egitim), "test şarkısı eğitimde!"
assert not any(f.startswith("cover_") or "simge" in f for f in egitim), "cover şarkısı eğitimde!"
assert all(f.startswith("dua_") for f in real) and all(f.startswith("synth_") for f in synth)
print(f"assert geçti: gerçek {len(real)} dosya, sentetik {len(synth)} dosya, test {len(test)} dosya")
for f in sorted(glob.glob("models/timings*.json")):
    for m, t in json.load(open(f)).items():
        print(f"{f}: {m} {t['slices']} dilim, {t['epochs']} epoch, {t['actual_steps']} adım, {t['sec_per_epoch']} sn/epoch")

**Modelin öğrenmesini dinle.** Aynı 10 saniyelik kıta, her modelin her checkpoint'iyle çevrilir (CPU'da birkaç dakika).
İlk checkpoint'lerde ses bulanıktır, sonra Dua Lipa'ya benzemeye başlar. Ses robotik ya da metalikleşmeye başladıysa model ezberlemiştir (overfitting):
**ondan önceki** checkpoint'i seç. Her satırın yanında o 10 saniyenin Ses benzerliği de var (kısa olduğu için sadece yön gösterir).

In [ ]:
sb = v.SesBenzerligi(glob.glob("dataset_temiz/test/*.wav"), HEDEF_VOKAL)
kita = str(OUT / "hedef_kita.wav")
satirlar = []
for m in ("A", "B"):
    idx = v.index_dosyasi(MODELLER[m])
    for ep, adim, pth in v.checkpointler(MODELLER[m]):
        out = OUT / "checkpoint" / f"{m}_{ep:03d}e.wav"
        if not out.exists():
            v.cevir(kita, out, pth, idx, pitch=KAYDIRMA)
        satirlar.append((m, ep, adim, str(out), sb.olc(out)["medyan"]))
html = "<table><tr><th>Model</th><th>Epoch</th><th>Adım</th><th>Dinle</th><th>Ses benzerliği</th></tr>"
for m, ep, adim, out, s in satirlar:
    html += f"<tr><td>{m}</td><td>{ep}</td><td>{adim}</td><td>{Audio(out)._repr_html_()}</td><td>%{s:.0f}</td></tr>"
display(HTML(html + "</table>"))

**Checkpoint seçimi.** Dinledikten sonra her model için seçtiğin epoch'u yaz (aynı adım sayısına yakın olanları seçmek adil karşılaştırmadır).
`None` bırakırsan son checkpoint kullanılır.

In [ ]:
SECIM = {"A": None, "B": None}
secilen = {}
for m, d in MODELLER.items():
    cks = v.checkpointler(d)
    ep, adim, pth = next(c for c in cks if c[0] == SECIM[m]) if SECIM[m] else cks[-1]
    secilen[m] = (ep, adim, pth, v.index_dosyasi(d))
    print(f"{m}: epoch {ep}, adım {adim}, {Path(pth).name}")

**Hedef şarkının tamamı.** Seçilen checkpoint'lerle kuru vokalin tamamı A ve B ile çevrilir (CPU'da modele göre birkaç dakika).
Öğretmen'in (Seed-VC) çevirisi GPU'dan hazır geldi (`teacher/`).

In [ ]:
tam = {"Öğretmen (Seed-VC, eğitimsiz)": f"teacher/{HEDEF}.wav"}
for m, (ep, adim, pth, idx) in secilen.items():
    out = OUT / f"{HEDEF}_{m}_{ep}e.wav"
    if not out.exists():
        v.cevir(HEDEF_VOKAL, out, pth, idx, pitch=KAYDIRMA)
    tam[m] = str(out)
for ad, f in tam.items():
    print(ad); display(Audio(f))

**Karşılaştırma tablosu.** Ses benzerliği: her 3 sn'lik pencere, orijinal şarkıcı (Simge) = %0 ve Dua Lipa'nın diğer test kaydı = %100 olacak şekilde ölçeklenir;
medyan ve %25–%75 aralığı. Oktav kontrolü: notaların yarım tondan yakın olma oranı, **%85'in altı** sorun demektir (genelde ani oktav atlamaları).
Uyarı: iki model de Öğretmen'den kötüyse eğitim datasında sorun vardır. ECAPA konuşmayla eğitildi; son kararı dinleme testi (Aşama 5) verir.

In [ ]:
test = sorted(glob.glob("dataset_temiz/test/*.wav"))
satir = [{"Satır": "Orijinal şarkıcı (Simge)", **sb.olc(HEDEF_VOKAL), "Oktav kontrolü": None},
         {"Satır": "Dua Lipa test kaydı (2. yarı)", **sb.olc(test[len(test) // 2:]), "Oktav kontrolü": None}]
for ad, f in tam.items():
    satir.append({"Satır": ad, **sb.olc(f), "Oktav kontrolü": v.oktav_kontrolu(HEDEF_VOKAL, f, KAYDIRMA)["yakin_yuzde"]})
tablo = pd.DataFrame(satir).set_index("Satır")
tablo = tablo.rename(columns={"medyan": "Ses benzerliği %", "p25": "%25", "p75": "%75"}).round(1)
tablo.to_csv(OUT / "karsilastirma.csv")
display(tablo)